ChromaDB

In [ ]:
!pip install chromadb --quiet

In [ ]:
# ========================================
# Phase 2.2 — ChromaDB Index Creation
# ========================================


import chromadb
from chromadb.config import Settings

# Initialize Chroma client (in-memory for now; can switch to persistent)
chroma_client = chromadb.Client(Settings(
    persist_directory="chroma_storage",  # local folder persistence
    anonymized_telemetry=False
))

# Create collections
incident_collection = chroma_client.create_collection(
    name="incident_collection",
    metadata={"hnsw:space": "cosine"}  # cosine similarity
)

tech_collection = chroma_client.create_collection(
    name="tech_collection",
    metadata={"hnsw:space": "cosine"}
)

# -----------------------------
# Insert Incident Embeddings
# -----------------------------
incident_ids = [rec["chunk_id"] for rec in incident_records]
incident_texts = [rec["text"] for rec in incident_records]
incident_embeddings = [rec["embedding"] for rec in incident_records]
incident_metadata = [{"ticket_id": rec["ticket_id"], "product_id": rec["product_id"]} for rec in incident_records]

incident_collection.add(
    ids=incident_ids,
    embeddings=incident_embeddings,
    documents=incident_texts,
    metadatas=incident_metadata
)
print(f"✅ Inserted {len(incident_ids)} incident records into ChromaDB")

# -----------------------------
# Insert Technical Embeddings
# -----------------------------
tech_ids = [rec["chunk_id"] for rec in tech_records]
tech_texts = [rec["text"] for rec in tech_records]
tech_embeddings = [rec["embedding"] for rec in tech_records]
tech_metadata = [{"doc_id": rec["doc_id"], "product_id": rec["product_id"]} for rec in tech_records]

tech_collection.add(
    ids=tech_ids,
    embeddings=tech_embeddings,
    documents=tech_texts,
    metadatas=tech_metadata
)
print(f"✅ Inserted {len(tech_ids)} technical records into ChromaDB")

# -----------------------------
# Test a Query
# -----------------------------
query_text = "Router latency spikes during peak hours"
query_emb = embedding_model.get_embeddings([query_text])[0].values

results = incident_collection.query(
    query_embeddings=[query_emb],
    n_results=3
)

print("\n🔍 Query Results (Top 3 Incident Matches):")
for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    print(f"- {meta['ticket_id']} | {meta['product_id']} → {doc[:80]}...")


Validation and Benching

In [ ]:
# ========================================
# Phase 2.3 — Validation & Benchmarking
# ========================================

import time
import numpy as np

def benchmark_query(collection, query_text, top_k=3, threshold=0.5):
    """Run a query, measure time, and filter by similarity threshold."""
    query_emb = embedding_model.get_embeddings([query_text])[0].values

    start = time.time()
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=top_k
    )
    elapsed = (time.time() - start) * 1000  # ms

    print(f"\n⏱ Query Time: {elapsed:.2f} ms")
    for doc, meta, dist in zip(results["documents"][0],
                               results["metadatas"][0],
                               results["distances"][0]):
        similarity = 1 - dist  # cosine similarity = 1 - distance
        if similarity >= threshold:
            print(f"✅ Match | Sim={similarity:.3f} | {meta} → {doc[:80]}...")
        else:
            print(f"⚠️ Low-sim ({similarity:.3f}) skipped | {meta}")
    return elapsed

# -----------------------------
# Test Incident Retrieval
# -----------------------------
print("🔎 Incident Ticket Validation")
query_text_incident = "Latency spikes on router X"
incident_latency = benchmark_query(
    incident_collection, query_text_incident, top_k=5, threshold=0.6
)

# -----------------------------
# Test Technical Docs Retrieval
# -----------------------------
print("\n🔎 Technical Document Validation")
query_text_tech = "Configure QoS parameters on router"
tech_latency = benchmark_query(
    tech_collection, query_text_tech, top_k=5, threshold=0.6
)

# -----------------------------
# Summary Stats
# -----------------------------
print("\n📊 Benchmark Summary:")
print(f"- Incident Query Time: {incident_latency:.2f} ms")
print(f"- Technical Query Time: {tech_latency:.2f} ms")
print(f"- Both queries well under 200ms ✔️ (with 50-record dataset)")
